<a href="https://colab.research.google.com/github/www-fidezDAE/fidez.RDS/blob/main/Brian_API_Clinical_Data_Extract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pandas requests tqdm

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
from tqdm.auto import tqdm

#Check the ClinicalTrials.gov API
url = "https://clinicaltrials.gov/api/v2/version"

response = requests.get(url, timeout=60)

print("Status code:", response.status_code)
print(response.json())

#Extract Uganda clinical trials first
import requests
import pandas as pd
import time

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "query.locn": "Uganda",
    "pageSize": 100,
    "format": "json"
}

response = requests.get(BASE_URL, params=params, timeout=60)

print("Status:", response.status_code)

data = response.json()

print("Number of records returned:",
      len(data.get("studies", [])))

#Download ALL Uganda studies
import requests
import pandas as pd
import time
from tqdm.auto import tqdm

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"


def download_trials(location="Uganda", page_size=100):

    all_studies = []
    page_token = None

    while True:

        params = {
            "query.locn": location,
            "pageSize": page_size,
            "format": "json"
        }

        if page_token:
            params["pageToken"] = page_token

        response = requests.get(
            BASE_URL,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        result = response.json()

        studies = result.get("studies", [])

        all_studies.extend(studies)

        print(
            f"Downloaded {len(studies)} records | "
            f"Total = {len(all_studies)}"
        )

        page_token = result.get("nextPageToken")

        if not page_token:
            break

        time.sleep(0.2)

    return all_studies


uganda_raw = download_trials("Uganda")

print("\nTotal Uganda records:", len(uganda_raw))

#Convert the JSON records into a usable dataset
def get_value(dictionary, *keys, default=np.nan):

    value = dictionary

    for key in keys:

        if isinstance(value, dict):
            value = value.get(key, default)

        else:
            return default

    return value

def extract_trial_data(studies):

    records = []

    for study in tqdm(studies):

        protocol = study.get(
            "protocolSection", {}
        )

        identification = protocol.get(
            "identificationModule", {}
        )

        status = protocol.get(
            "statusModule", {}
        )

        design = protocol.get(
            "designModule", {}
        )

        sponsor = protocol.get(
            "sponsorCollaboratorsModule", {}
        )

        contacts = protocol.get(
            "contactsLocationsModule", {}
        )

        arms = protocol.get(
            "armsInterventionsModule", {}
        )

        results = study.get(
            "resultsSection", {}
        )

        # Basic identification
        nct_id = identification.get(
            "nctId", np.nan
        )

        title = identification.get(
            "briefTitle", np.nan
        )

        # Dates
        start_date = get_value(
            status,
            "startDateStruct",
            "date"
        )

        completion_date = get_value(
            status,
            "completionDateStruct",
            "date"
        )

        study_first_posted = get_value(
            status,
            "studyFirstPostDateStruct",
            "date"
        )

        results_first_posted = get_value(
            status,
            "resultsFirstPostDateStruct",
            "date"
        )

        # Study status
        overall_status = status.get(
            "overallStatus", np.nan
        )

        # Design
        study_type = design.get(
            "studyType", np.nan
        )

        phases = design.get(
            "phases", []
        )

        phase = "; ".join(phases) if phases else np.nan

        allocation = design.get(
            "designInfo", {}
        ).get(
            "allocation", np.nan
        )

        intervention_model = design.get(
            "designInfo", {}
        ).get(
            "interventionModel", np.nan
        )

        masking_info = design.get(
            "designInfo", {}
        ).get(
            "maskingInfo", {}
        )

        masking = masking_info.get(
            "masking", np.nan
        )

        primary_purpose = design.get(
            "designInfo", {}
        ).get(
            "primaryPurpose", np.nan
        )

        # Enrollment
        enrollment_info = design.get(
            "enrollmentInfo", {}
        )

        target_enrollment = enrollment_info.get(
            "count", np.nan
        )

        enrollment_type = enrollment_info.get(
            "type", np.nan
        )

        # Sponsor
        lead_sponsor = sponsor.get(
            "leadSponsor", {}
        )

        sponsor_name = lead_sponsor.get(
            "name", np.nan
        )

        sponsor_class = lead_sponsor.get(
            "class", np.nan
        )

        # Locations
        locations = contacts.get(
            "locations", []
        )

        countries = set()
        facilities = set()

        for loc in locations:

            country = loc.get(
                "country"
            )

            facility = loc.get(
                "facility"
            )

            if country:
                countries.add(country)

            if facility:
                facilities.add(facility)

        number_countries = len(countries)

        number_sites = len(facilities)

        country_list = "; ".join(
            sorted(countries)
        )

        # Interventions
        interventions = arms.get(
            "interventions", []
        )

        intervention_types = []
        intervention_names = []

        for intervention in interventions:

            if intervention.get("type"):
                intervention_types.append(
                    intervention.get("type")
                )

            if intervention.get("name"):
                intervention_names.append(
                    intervention.get("name")
                )

        intervention_type = "; ".join(
            sorted(set(intervention_types))
        ) if intervention_types else np.nan

        intervention_name = "; ".join(
            sorted(set(intervention_names))
        ) if intervention_names else np.nan

        # Results
        has_results = bool(results)

        # Trial completion
        trial_completion = (
            1 if overall_status == "COMPLETED"
            else 0
        )

        record = {

            "NCT_ID": nct_id,

            "Study_Title": title,

            "Trial_Completion": trial_completion,

            "Study_Status": overall_status,

            "Results_Available": int(has_results),

            "Study_Type": study_type,

            "Study_Phase": phase,

            "Allocation": allocation,

            "Intervention_Model": intervention_model,

            "Masking": masking,

            "Primary_Purpose": primary_purpose,

            "Target_Enrollment": target_enrollment,

            "Enrollment_Type": enrollment_type,

            "Sponsor_Name": sponsor_name,

            "Sponsor_Type": sponsor_class,

            "Intervention_Type": intervention_type,

            "Intervention_Name": intervention_name,

            "Number_of_Sites": number_sites,

            "Number_of_Countries": number_countries,

            "Countries": country_list,

            "Start_Date": start_date,

            "Completion_Date": completion_date,

            "Study_First_Posted": study_first_posted,

            "Results_First_Posted": results_first_posted

        }

        records.append(record)

    return pd.DataFrame(records)

uganda_df = extract_trial_data(
    uganda_raw
)

print(
    "Final dataset:",
    uganda_df.shape
)

uganda_df.head()

#Calculate study duration
uganda_df["Start_Date"] = pd.to_datetime(
    uganda_df["Start_Date"],
    errors="coerce"
)

uganda_df["Completion_Date"] = pd.to_datetime(
    uganda_df["Completion_Date"],
    errors="coerce"
)

uganda_df["Results_First_Posted"] = pd.to_datetime(
    uganda_df["Results_First_Posted"],
    errors="coerce"
)

uganda_df["Study_First_Posted"] = pd.to_datetime(
    uganda_df["Study_First_Posted"],
    errors="coerce"
)

uganda_df["Study_Duration_Months"] = (
    (uganda_df["Completion_Date"] -
     uganda_df["Start_Date"])
    .dt.days / 30.4375
)

#Calculate time to results reporting
uganda_df["Days_to_Results_Reporting"] = (
    uganda_df["Results_First_Posted"] -
    uganda_df["Completion_Date"]
).dt.days

uganda_df[
    [
        "NCT_ID",
        "Study_Status",
        "Completion_Date",
        "Results_First_Posted",
        "Days_to_Results_Reporting"
    ]
].head(20)

#Create the results availability variable
uganda_df["Results_Available"] = (
    uganda_df["Results_First_Posted"]
    .notna()
    .astype(int)
)

#Create a preliminary timely reporting variable
uganda_df["Timely_Results_Reporting"] = np.where(

    uganda_df["Days_to_Results_Reporting"].notna() &
    (uganda_df["Days_to_Results_Reporting"] <= 365),

    1,

    0
)

#Clean the dataset
uganda_df = uganda_df.drop_duplicates(
    subset="NCT_ID"
)

uganda_df = uganda_df.reset_index(
    drop=True
)

print("Number of unique trials:",
      uganda_df["NCT_ID"].nunique())

print("Number of variables:",
      uganda_df.shape[1])

missing = pd.DataFrame({

    "Variable": uganda_df.columns,

    "Missing_Count":
        uganda_df.isna().sum().values,

    "Missing_Percentage":
        (
            uganda_df.isna().mean().values * 100
        ).round(2)

})

missing.sort_values(
    "Missing_Percentage",
    ascending=False
)

#Save the Uganda dataset as CSV
file_name = "Brian_Kabuubi_Clinical_Trials_Uganda.csv"

uganda_df.to_csv(
    file_name,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Dataset saved as: {file_name}"
)

from google.colab import files

files.download(
    file_name
)

Status code: 200
{'apiVersion': '2.0.5', 'dataTimestamp': '2026-08-28T09:00:06'}
Status: 200
Number of records returned: 100
Downloaded 100 records | Total = 100
Downloaded 100 records | Total = 200
Downloaded 100 records | Total = 300
Downloaded 100 records | Total = 400
Downloaded 100 records | Total = 500
Downloaded 100 records | Total = 600
Downloaded 100 records | Total = 700
Downloaded 100 records | Total = 800
Downloaded 100 records | Total = 900
Downloaded 72 records | Total = 972

Total Uganda records: 972


  0%|          | 0/972 [00:00<?, ?it/s]

Final dataset: (972, 24)
Number of unique trials: 972
Number of variables: 27
Dataset saved as: Brian_Kabuubi_Clinical_Trials_Uganda.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Correct Uganda Specific API Data Extraction Pipeline

In [1]:
# ================================================================
# BRIAN KABUUBI
# REGISTRATION NUMBER: 2026 04 47856
# CEM, KAMPALA INTERNATIONAL UNIVERSITY
#
# THESIS:
# DETERMINANTS OF CLINICAL TRIAL COMPLETION AND RESULTS REPORTING
# AMONG TRIALS INVOLVING UGANDA:
# EVIDENCE FROM CLINICALTRIALS.GOV
#
# GENERAL OBJECTIVE:
# To assess the determinants of clinical trial completion and results
# reporting among trials involving Uganda using registered clinical
# trial records from ClinicalTrials.gov.
#
# SINGLE GOOGLE COLAB PIPELINE
# ================================================================


# ================================================================
# 1. INSTALL / IMPORT REQUIRED PACKAGES
# ================================================================

import requests
import pandas as pd
import numpy as np
import json
import re
import os
import time
from datetime import datetime
from collections import Counter
from google.colab import files

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)


# ================================================================
# 2. STUDY INFORMATION
# ================================================================

STUDENT_NAME = "Brian Kabuubi"
REG_NO = "2026 04 47856"
COLLEGE = "CEM, Kampala International University"

THESIS_TITLE = (
    "DETERMINANTS OF CLINICAL TRIAL COMPLETION AND RESULTS REPORTING "
    "AMONG TRIALS INVOLVING UGANDA: EVIDENCE FROM CLINICALTRIALS.GOV"
)

API_BASE = "https://clinicaltrials.gov/api/v2/studies"

print("=" * 80)
print("CLINICALTRIALS.GOV DATA EXTRACTION PIPELINE")
print("=" * 80)
print(f"Student: {STUDENT_NAME}")
print(f"Registration Number: {REG_NO}")
print(f"College: {COLLEGE}")
print(f"Location: Uganda")
print("=" * 80)


# ================================================================
# 3. CHECK CLINICALTRIALS.GOV API VERSION / DATA TIMESTAMP
# ================================================================

version_url = "https://clinicaltrials.gov/api/v2/version"

version_response = requests.get(version_url, timeout=60)
version_response.raise_for_status()

version_data = version_response.json()

print("\nClinicalTrials.gov API information")
print("-" * 80)
print(json.dumps(version_data, indent=2))


# ================================================================
# 4. FUNCTION FOR SAFE EXTRACTION FROM NESTED JSON
# ================================================================

def get_nested(data, *keys, default=None):
    """
    Safely retrieve a nested value from a dictionary.
    Returns default if a key does not exist.
    """
    current = data

    for key in keys:

        if isinstance(current, dict) and key in current:
            current = current[key]

        else:
            return default

    return current


def first_nonempty(*values):
    """
    Return the first nonempty value.
    """
    for value in values:

        if value is not None and value != "" and value != []:
            return value

    return None


def clean_text(value):
    """
    Convert values to clean text.
    """
    if value is None:
        return ""

    if isinstance(value, list):
        return "; ".join([str(x) for x in value if x is not None])

    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    return str(value).strip()


# ================================================================
# 5. API PAGINATION FUNCTION
# ================================================================

def retrieve_all_studies(location="Uganda", page_size=100):
    """
    Retrieve all ClinicalTrials.gov studies involving Uganda.

    The modern API returns a nextPageToken when more records remain.
    """

    all_studies = []
    next_token = None
    page_number = 1

    while True:

        params = {
            "query.locn": location,
            "pageSize": page_size,
            "format": "json"
        }

        if next_token:
            params["pageToken"] = next_token

        print(f"\nDownloading page {page_number} ...")

        response = requests.get(
            API_BASE,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        studies = data.get("studies", [])

        print(f"Records retrieved on this page: {len(studies)}")

        all_studies.extend(studies)

        next_token = data.get("nextPageToken")

        if not next_token:
            break

        page_number += 1

        # Small pause to avoid unnecessary request pressure
        time.sleep(0.5)

    print("\n" + "=" * 80)
    print(f"TOTAL RAW RECORDS RETRIEVED: {len(all_studies):,}")
    print("=" * 80)

    return all_studies


# ================================================================
# 6. RETRIEVE THE UGANDA DATA
# ================================================================

raw_studies = retrieve_all_studies(
    location="Uganda",
    page_size=100
)


# ================================================================
# 7. BASIC DUPLICATE CHECK
# ================================================================

raw_ids = []

for study in raw_studies:

    nct_id = get_nested(
        study,
        "protocolSection",
        "identificationModule",
        "nctId"
    )

    if nct_id:
        raw_ids.append(nct_id)

print(f"\nNumber of NCT IDs retrieved: {len(raw_ids):,}")
print(f"Number of unique NCT IDs: {len(set(raw_ids)):,}")
print(f"Duplicate records: {len(raw_ids) - len(set(raw_ids)):,}")


# ================================================================
# 8. FUNCTION TO EXTRACT ONE STUDY
# ================================================================

def extract_study(study):

    protocol = study.get("protocolSection", {})
    results = study.get("resultsSection", {})

    identification = protocol.get(
        "identificationModule", {}
    )

    status = protocol.get(
        "statusModule", {}
    )

    design = protocol.get(
        "designModule", {}
    )

    sponsor = protocol.get(
        "sponsorCollaboratorsModule", {}
    )

    contacts = protocol.get(
        "contactsLocationsModule", {}
    )

    arms = protocol.get(
        "armsInterventionsModule", {}
    )

    description = protocol.get(
        "descriptionModule", {}
    )

    # ------------------------------------------------------------
    # IDENTIFICATION
    # ------------------------------------------------------------

    nct_id = identification.get("nctId")

    brief_title = identification.get(
        "briefTitle"
    )

    official_title = identification.get(
        "officialTitle"
    )

    study_type = design.get(
        "studyType"
    )

    # ------------------------------------------------------------
    # STATUS
    # ------------------------------------------------------------

    overall_status = status.get(
        "overallStatus"
    )

    start_date_struct = status.get(
        "startDateStruct",
        {}
    )

    completion_date_struct = status.get(
        "completionDateStruct",
        {}
    )

    primary_completion_date_struct = status.get(
        "primaryCompletionDateStruct",
        {}
    )

    study_start_date = start_date_struct.get(
        "date"
    )

    study_completion_date = completion_date_struct.get(
        "date"
    )

    primary_completion_date = primary_completion_date_struct.get(
        "date"
    )

    study_start_type = start_date_struct.get(
        "type"
    )

    study_completion_type = completion_date_struct.get(
        "type"
    )

    primary_completion_type = primary_completion_date_struct.get(
        "type"
    )

    # ------------------------------------------------------------
    # DESIGN
    # ------------------------------------------------------------

    phases = design.get(
        "phases",
        []
    )

    phase = "; ".join(phases) if phases else ""

    allocation = design.get(
        "allocation"
    )

    intervention_model = design.get(
        "interventionModel"
    )

    intervention_model_description = design.get(
        "interventionModelDescription"
    )

    primary_purpose = design.get(
        "primaryPurpose"
    )

    masking_info = design.get(
        "maskingInfo",
        {}
    )

    masking = masking_info.get(
        "masking"
    )

    masking_description = masking_info.get(
        "maskingDescription"
    )

    enrollment_info = design.get(
        "enrollmentInfo",
        {}
    )

    enrollment_count = enrollment_info.get(
        "count"
    )

    enrollment_type = enrollment_info.get(
        "type"
    )

    # ------------------------------------------------------------
    # SPONSOR
    # ------------------------------------------------------------

    lead_sponsor = sponsor.get(
        "leadSponsor",
        {}
    )

    sponsor_name = lead_sponsor.get(
        "name"
    )

    sponsor_class = lead_sponsor.get(
        "class"
    )

    # ------------------------------------------------------------
    # LOCATIONS
    # ------------------------------------------------------------

    locations = contacts.get(
        "locations",
        []
    )

    countries = []

    facilities = []

    uganda_locations = []

    for location in locations:

        country = location.get(
            "country"
        )

        facility = location.get(
            "facility"
        )

        if country:
            countries.append(country)

        if facility:
            facilities.append(facility)

        if country and country.strip().lower() == "uganda":

            uganda_locations.append(
                location
            )

    unique_countries = sorted(
        set(
            [x for x in countries if x]
        )
    )

    unique_facilities = sorted(
        set(
            [x for x in facilities if x]
        )
    )

    # ------------------------------------------------------------
    # INTERVENTIONS
    # ------------------------------------------------------------

    interventions = arms.get(
        "interventions",
        []
    )

    intervention_types = []
    intervention_names = []

    for intervention in interventions:

        intervention_type = intervention.get(
            "type"
        )

        intervention_name = intervention.get(
            "name"
        )

        if intervention_type:
            intervention_types.append(
                intervention_type
            )

        if intervention_name:
            intervention_names.append(
                intervention_name
            )

    intervention_types = sorted(
        set(intervention_types)
    )

    intervention_names = sorted(
        set(intervention_names)
    )

    # ------------------------------------------------------------
    # RESULTS
    # ------------------------------------------------------------

    results_info = results.get(
        "studyType"
    )

    results_first_posted = status.get(
        "resultsFirstPostDateStruct",
        {}
    ).get(
        "date"
    )

    results_first_posted_type = status.get(
        "resultsFirstPostDateStruct",
        {}
    ).get(
        "type"
    )

    results_first_posted_qc = status.get(
        "resultsFirstPostDateStruct",
        {}
    ).get(
        "date"
    )

    results_first_submitted = status.get(
        "resultsFirstSubmitDate"
    )

    results_first_posted_with_qc = status.get(
        "resultsFirstPostDateStruct",
        {}
    ).get(
        "date"
    )

    # ------------------------------------------------------------
    # RESULTS SECTION CHECK
    # ------------------------------------------------------------

    has_results_section = bool(
        results
    )

    # ------------------------------------------------------------
    # DATE CONVERSION
    # ------------------------------------------------------------

    start_dt = pd.to_datetime(
        study_start_date,
        errors="coerce"
    )

    completion_dt = pd.to_datetime(
        study_completion_date,
        errors="coerce"
    )

    primary_completion_dt = pd.to_datetime(
        primary_completion_date,
        errors="coerce"
    )

    results_posted_dt = pd.to_datetime(
        results_first_posted,
        errors="coerce"
    )

    # ------------------------------------------------------------
    # STUDY DURATION
    # ------------------------------------------------------------

    if (
        pd.notna(start_dt)
        and pd.notna(completion_dt)
        and completion_dt >= start_dt
    ):

        study_duration_days = (
            completion_dt - start_dt
        ).days

        study_duration_months = (
            study_duration_days / 30.4375
        )

    else:

        study_duration_days = np.nan
        study_duration_months = np.nan

    # ------------------------------------------------------------
    # DAYS FROM PRIMARY COMPLETION TO RESULTS POSTING
    # ------------------------------------------------------------

    if (
        pd.notna(primary_completion_dt)
        and pd.notna(results_posted_dt)
    ):

        days_to_results = (
            results_posted_dt
            - primary_completion_dt
        ).days

    else:

        days_to_results = np.nan

    # ------------------------------------------------------------
    # TRIAL COMPLETION
    # ------------------------------------------------------------

    trial_completion = (
        1
        if str(overall_status).upper() == "COMPLETED"
        else 0
    )

    # ------------------------------------------------------------
    # RESULTS AVAILABILITY
    #
    # Results are treated as available when a valid
    # Results First Posted date exists.
    # ------------------------------------------------------------

    results_available = (
        1
        if pd.notna(results_posted_dt)
        else 0
    )

    # ------------------------------------------------------------
    # TIMELY RESULTS REPORTING
    #
    # IMPORTANT:
    #
    # We DO NOT classify missing dates as delayed.
    #
    # Timely = 1 if results were posted within 365 days
    # of primary completion.
    #
    # Delayed = 0 if results were posted after 365 days.
    #
    # Missing = NaN if either required date is unavailable.
    # ------------------------------------------------------------

    if pd.notna(days_to_results):

        if days_to_results < 0:

            timely_results_reporting = np.nan

        elif days_to_results <= 365:

            timely_results_reporting = 1

        else:

            timely_results_reporting = 0

    else:

        timely_results_reporting = np.nan

    # ------------------------------------------------------------
    # REPORTING CATEGORY
    # ------------------------------------------------------------

    if pd.isna(days_to_results):

        reporting_category = "Not assessable"

    elif days_to_results < 0:

        reporting_category = "Date inconsistency"

    elif days_to_results <= 365:

        reporting_category = "Timely"

    else:

        reporting_category = "Delayed"

    # ------------------------------------------------------------
    # SPONSOR CLASSIFICATION
    # ------------------------------------------------------------

    sponsor_class_upper = str(
        sponsor_class
    ).upper()

    sponsor_name_upper = str(
        sponsor_name
    ).upper()

    if sponsor_class_upper == "INDUSTRY":

        sponsor_group = "Industry"

    elif sponsor_class_upper in [
        "NIH",
        "U.S. FEDERAL AGENCY"
    ]:

        sponsor_group = "Government"

    elif sponsor_class_upper in [
        "OTHER",
        "NETWORK"
    ]:

        sponsor_group = "Academic/Other"

    elif (
        "UNIVERSITY" in sponsor_name_upper
        or "COLLEGE" in sponsor_name_upper
        or "INSTITUTE" in sponsor_name_upper
        or "SCHOOL" in sponsor_name_upper
        or "HOSPITAL" in sponsor_name_upper
    ):

        sponsor_group = "Academic/Health Institution"

    elif sponsor_class_upper:

        sponsor_group = sponsor_class_upper.title()

    else:

        sponsor_group = "Unknown"

    # ------------------------------------------------------------
    # INTERVENTION CATEGORY
    # ------------------------------------------------------------

    if not intervention_types:

        intervention_category = "Not applicable"

    elif len(intervention_types) == 1:

        intervention_category = intervention_types[0]

    else:

        intervention_category = "Multiple"

    # ------------------------------------------------------------
    # PHASE CATEGORY
    # ------------------------------------------------------------

    if not phase:

        phase_category = "Not applicable"

    elif "PHASE4" in phase.upper():

        phase_category = "Phase 4"

    elif "PHASE3" in phase.upper():

        phase_category = "Phase 3"

    elif "PHASE2" in phase.upper():

        phase_category = "Phase 2"

    elif "PHASE1" in phase.upper():

        phase_category = "Phase 1"

    elif "EARLY_PHASE1" in phase.upper():

        phase_category = "Early Phase 1"

    else:

        phase_category = phase

    # ------------------------------------------------------------
    # COUNTRY COUNT
    # ------------------------------------------------------------

    number_of_countries = len(
        unique_countries
    )

    # ------------------------------------------------------------
    # SITE COUNT
    # ------------------------------------------------------------

    number_of_sites = len(
        unique_facilities
    )

    # ------------------------------------------------------------
    # MULTINATIONAL STUDY
    # ------------------------------------------------------------

    multinational_study = (
        1
        if number_of_countries > 1
        else 0
    )

    # ------------------------------------------------------------
    # MULTISITE STUDY
    # ------------------------------------------------------------

    multisite_study = (
        1
        if number_of_sites > 1
        else 0
    )

    # ------------------------------------------------------------
    # UGANDA SITE COUNT
    # ------------------------------------------------------------

    number_of_uganda_sites = len(
        uganda_locations
    )

    # ------------------------------------------------------------
    # FINAL RECORD
    # ------------------------------------------------------------

    return {

        # Identification
        "NCT_ID": nct_id,
        "Brief_Title": brief_title,
        "Official_Title": official_title,

        # Study status
        "Study_Type": study_type,
        "Overall_Status": overall_status,

        # Completion outcomes
        "Trial_Completion": trial_completion,
        "Results_Available": results_available,
        "Timely_Results_Reporting": timely_results_reporting,

        # Reporting
        "Reporting_Category": reporting_category,
        "Days_to_Results_Reporting": days_to_results,

        # Dates
        "Study_Start_Date": study_start_date,
        "Primary_Completion_Date": primary_completion_date,
        "Study_Completion_Date": study_completion_date,
        "Results_First_Posted_Date": results_first_posted,

        # Date types
        "Study_Start_Date_Type": study_start_type,
        "Primary_Completion_Date_Type": primary_completion_type,
        "Study_Completion_Date_Type": study_completion_type,
        "Results_First_Posted_Date_Type": results_first_posted_type,

        # Design
        "Phase": phase,
        "Phase_Category": phase_category,
        "Allocation": allocation,
        "Intervention_Model": intervention_model,
        "Intervention_Model_Description": intervention_model_description,
        "Masking": masking,
        "Masking_Description": masking_description,
        "Primary_Purpose": primary_purpose,

        # Enrolment
        "Target_Enrolment": enrollment_count,
        "Enrolment_Type": enrollment_type,

        # Sponsor
        "Sponsor_Name": sponsor_name,
        "Sponsor_Class": sponsor_class,
        "Sponsor_Group": sponsor_group,

        # Intervention
        "Intervention_Type": intervention_category,
        "Intervention_Types": "; ".join(intervention_types),
        "Intervention_Names": "; ".join(intervention_names),

        # Geographic information
        "Countries": "; ".join(unique_countries),
        "Number_of_Countries": number_of_countries,
        "Multinational_Study": multinational_study,

        "Number_of_Sites": number_of_sites,
        "Multisite_Study": multisite_study,

        "Number_of_Uganda_Sites": number_of_uganda_sites,

        # Duration
        "Study_Duration_Days": study_duration_days,
        "Study_Duration_Months": study_duration_months,

        # Metadata
        "Uganda_Involvement": 1,
        "Data_Source": "ClinicalTrials.gov",
        "Extraction_Location": "Uganda"
    }


# ================================================================
# 9. EXTRACT ALL STUDIES
# ================================================================

print("\nExtracting study variables...")

records = []

for i, study in enumerate(raw_studies):

    try:

        record = extract_study(study)

        records.append(record)

    except Exception as e:

        print(
            f"Error extracting record {i}: {e}"
        )

print(
    f"\nSuccessfully extracted {len(records):,} records."
)


# ================================================================
# 10. CREATE DATAFRAME
# ================================================================

df = pd.DataFrame(records)


# ================================================================
# 11. REMOVE DUPLICATE NCT RECORDS
# ================================================================

before_duplicates = len(df)

df = df.drop_duplicates(
    subset=["NCT_ID"]
).reset_index(drop=True)

after_duplicates = len(df)

print("\nDuplicate check")
print("-" * 80)
print(f"Records before duplicate removal: {before_duplicates:,}")
print(f"Records after duplicate removal:  {after_duplicates:,}")
print(f"Duplicates removed: {before_duplicates - after_duplicates:,}")


# ================================================================
# 12. CORRECT DATA TYPES
# ================================================================

numeric_variables = [

    "Trial_Completion",
    "Results_Available",
    "Timely_Results_Reporting",
    "Days_to_Results_Reporting",
    "Target_Enrolment",
    "Number_of_Countries",
    "Number_of_Sites",
    "Number_of_Uganda_Sites",
    "Multinational_Study",
    "Multisite_Study",
    "Study_Duration_Days",
    "Study_Duration_Months"
]

for variable in numeric_variables:

    if variable in df.columns:

        df[variable] = pd.to_numeric(
            df[variable],
            errors="coerce"
        )


# ================================================================
# 13. DATE VARIABLES
# ================================================================

date_variables = [

    "Study_Start_Date",
    "Primary_Completion_Date",
    "Study_Completion_Date",
    "Results_First_Posted_Date"
]

for variable in date_variables:

    df[variable] = pd.to_datetime(
        df[variable],
        errors="coerce"
    )


# ================================================================
# 14. IDENTIFY DATE INCONSISTENCIES
# ================================================================

df["Date_Inconsistency"] = 0

condition_negative = (
    df["Days_to_Results_Reporting"] < 0
)

df.loc[
    condition_negative,
    "Date_Inconsistency"
] = 1


# ================================================================
# 15. CREATE ANALYSIS ELIGIBILITY VARIABLES
# ================================================================

# Eligible for Objective Two
# Trials with known completion status
df["Eligible_Objective_2"] = (
    df["Overall_Status"].notna()
).astype(int)


# Eligible for results availability analysis
df["Eligible_Results_Analysis"] = (
    df["Trial_Completion"] == 1
).astype(int)


# Eligible for timely reporting analysis
#
# Requires:
# 1. Completed trial
# 2. Primary completion date
# 3. Results first posted date
# 4. Nonnegative reporting interval
#
df["Eligible_Timely_Reporting"] = (
    (df["Trial_Completion"] == 1)
    &
    (df["Primary_Completion_Date"].notna())
    &
    (df["Results_First_Posted_Date"].notna())
    &
    (df["Days_to_Results_Reporting"] >= 0)
).astype(int)


# ================================================================
# 16. CREATE TARGET ENROLMENT CATEGORIES
# ================================================================

def enrolment_category(x):

    if pd.isna(x):
        return "Missing"

    if x < 100:
        return "<100"

    elif x < 500:
        return "100 to 499"

    elif x < 1000:
        return "500 to 999"

    else:
        return "1000+"


df["Target_Enrolment_Category"] = (
    df["Target_Enrolment"]
    .apply(enrolment_category)
)


# ================================================================
# 17. CREATE STUDY DURATION CATEGORIES
# ================================================================

def duration_category(x):

    if pd.isna(x):
        return "Missing"

    if x < 12:
        return "<12 months"

    elif x < 24:
        return "12 to <24 months"

    elif x < 36:
        return "24 to <36 months"

    else:
        return "36+ months"


df["Study_Duration_Category"] = (
    df["Study_Duration_Months"]
    .apply(duration_category)
)


# ================================================================
# 18. STANDARDIZE TEXT VARIABLES
# ================================================================

text_variables = [

    "Study_Type",
    "Overall_Status",
    "Phase_Category",
    "Allocation",
    "Intervention_Model",
    "Masking",
    "Primary_Purpose",
    "Sponsor_Group",
    "Intervention_Type",
    "Reporting_Category"
]

for variable in text_variables:

    if variable in df.columns:

        df[variable] = (
            df[variable]
            .fillna("Missing")
            .astype(str)
            .str.strip()
        )


# ================================================================
# 19. SORT DATA
# ================================================================

df = df.sort_values(
    by="NCT_ID"
).reset_index(drop=True)


# ================================================================
# 20. DISPLAY BASIC DATASET INFORMATION
# ================================================================

print("\n" + "=" * 80)
print("FINAL DATASET INFORMATION")
print("=" * 80)

print(
    f"Number of observations: {len(df):,}"
)

print(
    f"Number of variables: {len(df.columns):,}"
)

print(
    f"Unique NCT IDs: {df['NCT_ID'].nunique():,}"
)


# ================================================================
# 21. DESCRIPTIVE SUMMARY FOR OBJECTIVE ONE
# ================================================================

print("\n" + "=" * 80)
print("OBJECTIVE ONE: TRIAL STATUS")
print("=" * 80)

status_summary = (
    df["Overall_Status"]
    .value_counts(dropna=False)
    .rename_axis("Trial_Status")
    .reset_index(
        name="Frequency"
    )
)

status_summary["Percentage"] = (
    status_summary["Frequency"]
    / len(df)
    * 100
).round(2)

print(status_summary)


# ================================================================
# 22. COMPLETION SUMMARY
# ================================================================

completion_summary = pd.DataFrame({

    "Outcome": [
        "Completed",
        "Not Completed"
    ],

    "Frequency": [
        int(
            (df["Trial_Completion"] == 1)
            .sum()
        ),
        int(
            (df["Trial_Completion"] == 0)
            .sum()
        )
    ]
})

completion_summary["Percentage"] = (
    completion_summary["Frequency"]
    / len(df)
    * 100
).round(2)

print("\nTrial completion")
print(completion_summary)


# ================================================================
# 23. RESULTS AVAILABILITY SUMMARY
# ================================================================

results_summary = pd.DataFrame({

    "Results_Status": [
        "Results Available",
        "Results Not Available"
    ],

    "Frequency": [
        int(
            (df["Results_Available"] == 1)
            .sum()
        ),
        int(
            (df["Results_Available"] == 0)
            .sum()
        )
    ]
})

results_summary["Percentage"] = (
    results_summary["Frequency"]
    / len(df)
    * 100
).round(2)

print("\nResults availability")
print(results_summary)


# ================================================================
# 24. TIMELY REPORTING SUMMARY
# ================================================================

timely_df = df[
    df["Eligible_Timely_Reporting"] == 1
].copy()

timely_summary = pd.DataFrame({

    "Reporting_Status": [
        "Timely",
        "Delayed"
    ],

    "Frequency": [
        int(
            (
                timely_df[
                    "Timely_Results_Reporting"
                ] == 1
            ).sum()
        ),

        int(
            (
                timely_df[
                    "Timely_Results_Reporting"
                ] == 0
            ).sum()
        )
    ]
})

if len(timely_df) > 0:

    timely_summary["Percentage"] = (
        timely_summary["Frequency"]
        / len(timely_df)
        * 100
    ).round(2)

else:

    timely_summary["Percentage"] = np.nan

print("\nTimely results reporting")
print(timely_summary)


# ================================================================
# 25. SPONSOR SUMMARY
# ================================================================

sponsor_summary = (
    df["Sponsor_Group"]
    .value_counts(dropna=False)
    .rename_axis("Sponsor_Group")
    .reset_index(
        name="Frequency"
    )
)

sponsor_summary["Percentage"] = (
    sponsor_summary["Frequency"]
    / len(df)
    * 100
).round(2)

print("\nSponsor group")
print(sponsor_summary)


# ================================================================
# 26. STUDY TYPE SUMMARY
# ================================================================

study_type_summary = (
    df["Study_Type"]
    .value_counts(dropna=False)
    .rename_axis("Study_Type")
    .reset_index(
        name="Frequency"
    )
)

study_type_summary["Percentage"] = (
    study_type_summary["Frequency"]
    / len(df)
    * 100
).round(2)

print("\nStudy type")
print(study_type_summary)


# ================================================================
# 27. PHASE SUMMARY
# ================================================================

phase_summary = (
    df["Phase_Category"]
    .value_counts(dropna=False)
    .rename_axis("Study_Phase")
    .reset_index(
        name="Frequency"
    )
)

phase_summary["Percentage"] = (
    phase_summary["Frequency"]
    / len(df)
    * 100
).round(2)

print("\nStudy phase")
print(phase_summary)


# ================================================================
# 28. MISSING DATA ANALYSIS
# ================================================================

missing_summary = pd.DataFrame({

    "Variable": df.columns,

    "Missing_Frequency": [
        df[column].isna().sum()
        for column in df.columns
    ],

    "Total_Observations": [
        len(df)
        for _ in df.columns
    ]
})

missing_summary["Missing_Percentage"] = (
    missing_summary["Missing_Frequency"]
    / len(df)
    * 100
).round(2)

missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 80)
print("MISSING DATA ANALYSIS")
print("=" * 80)

display(
    missing_summary.head(30)
)


# ================================================================
# 29. MISSING DATA FOR PRINCIPAL ANALYTICAL VARIABLES
# ================================================================

principal_variables = [

    "Trial_Completion",
    "Results_Available",
    "Timely_Results_Reporting",
    "Study_Type",
    "Phase_Category",
    "Allocation",
    "Intervention_Model",
    "Masking",
    "Primary_Purpose",
    "Target_Enrolment",
    "Sponsor_Group",
    "Intervention_Type",
    "Number_of_Sites",
    "Number_of_Countries",
    "Study_Duration_Months",
    "Primary_Completion_Date",
    "Results_First_Posted_Date"
]

principal_missing = missing_summary[
    missing_summary["Variable"].isin(
        principal_variables
    )
].copy()

print(
    "\nMissingness among principal analytical variables"
)

display(
    principal_missing
)


# ================================================================
# 30. DATA QUALITY CHECKS
# ================================================================

quality_checks = {}

quality_checks[
    "Duplicate_NCT_IDs"
] = int(
    df["NCT_ID"].duplicated().sum()
)

quality_checks[
    "Negative_Reporting_Intervals"
] = int(
    (
        df["Days_to_Results_Reporting"] < 0
    ).sum()
)

quality_checks[
    "Completed_Trials"
] = int(
    (
        df["Trial_Completion"] == 1
    ).sum()
)

quality_checks[
    "Completed_Trials_With_Results"
] = int(
    (
        (df["Trial_Completion"] == 1)
        &
        (df["Results_Available"] == 1)
    ).sum()
)

quality_checks[
    "Completed_Trials_Eligible_For_Timely_Analysis"
] = int(
    (
        df["Eligible_Timely_Reporting"] == 1
    ).sum()
)

quality_checks[
    "Timely_Results"
] = int(
    (
        df["Timely_Results_Reporting"] == 1
    ).sum()
)

quality_checks[
    "Delayed_Results"
] = int(
    (
        df["Timely_Results_Reporting"] == 0
    ).sum()
)

quality_checks_df = pd.DataFrame(
    list(
        quality_checks.items()
    ),
    columns=[
        "Quality_Check",
        "Frequency"
    ]
)

print("\n" + "=" * 80)
print("DATA QUALITY CHECKS")
print("=" * 80)

display(
    quality_checks_df
)


# ================================================================
# 31. DESCRIPTIVE STATISTICS FOR NUMERIC VARIABLES
# ================================================================

numeric_descriptive_variables = [

    "Target_Enrolment",
    "Number_of_Sites",
    "Number_of_Uganda_Sites",
    "Number_of_Countries",
    "Study_Duration_Months",
    "Days_to_Results_Reporting"
]

descriptive_statistics = (
    df[
        numeric_descriptive_variables
    ]
    .describe()
    .T
)

descriptive_statistics[
    "Missing"
] = (
    df[
        numeric_descriptive_variables
    ]
    .isna()
    .sum()
)

print("\n" + "=" * 80)
print("NUMERIC DESCRIPTIVE STATISTICS")
print("=" * 80)

display(
    descriptive_statistics
)


# ================================================================
# 32. CROSS TABULATION:
#     STUDY TYPE BY COMPLETION
# ================================================================

study_type_completion = pd.crosstab(

    df["Study_Type"],

    df["Trial_Completion"],

    margins=True
)

study_type_completion.columns = [
    "Not_Completed",
    "Completed",
    "Total"
]

print("\nStudy type by completion")
display(
    study_type_completion
)


# ================================================================
# 33. CROSS TABULATION:
#     SPONSOR GROUP BY COMPLETION
# ================================================================

sponsor_completion = pd.crosstab(

    df["Sponsor_Group"],

    df["Trial_Completion"],

    margins=True
)

sponsor_completion.columns = [
    "Not_Completed",
    "Completed",
    "Total"
]

print("\nSponsor group by completion")
display(
    sponsor_completion
)


# ================================================================
# 34. CROSS TABULATION:
#     PHASE BY COMPLETION
# ================================================================

phase_completion = pd.crosstab(

    df["Phase_Category"],

    df["Trial_Completion"],

    margins=True
)

phase_completion.columns = [
    "Not_Completed",
    "Completed",
    "Total"
]

print("\nStudy phase by completion")
display(
    phase_completion
)


# ================================================================
# 35. CROSS TABULATION:
#     SPONSOR GROUP BY RESULTS AVAILABILITY
# ================================================================

sponsor_results = pd.crosstab(

    df["Sponsor_Group"],

    df["Results_Available"],

    margins=True
)

sponsor_results.columns = [
    "Results_Not_Available",
    "Results_Available",
    "Total"
]

print("\nSponsor group by results availability")
display(
    sponsor_results
)


# ================================================================
# 36. CROSS TABULATION:
#     PHASE BY RESULTS AVAILABILITY
# ================================================================

phase_results = pd.crosstab(

    df["Phase_Category"],

    df["Results_Available"],

    margins=True
)

phase_results.columns = [
    "Results_Not_Available",
    "Results_Available",
    "Total"
]

print("\nStudy phase by results availability")
display(
    phase_results
)


# ================================================================
# 37. CROSS TABULATION:
#     SPONSOR GROUP BY TIMELY REPORTING
# ================================================================

reporting_analysis = df[
    df["Eligible_Timely_Reporting"] == 1
].copy()

if len(reporting_analysis) > 0:

    sponsor_reporting = pd.crosstab(

        reporting_analysis[
            "Sponsor_Group"
        ],

        reporting_analysis[
            "Timely_Results_Reporting"
        ],

        margins=True
    )

    sponsor_reporting.columns = [
        "Delayed",
        "Timely",
        "Total"
    ]

    print(
        "\nSponsor group by timely results reporting"
    )

    display(
        sponsor_reporting
    )

else:

    sponsor_reporting = pd.DataFrame()

    print(
        "\nNo eligible observations for timely reporting analysis."
    )


# ================================================================
# 38. CREATE A COMPLETE DATA DICTIONARY
# ================================================================

data_dictionary = pd.DataFrame({

    "Variable": [

        "NCT_ID",
        "Brief_Title",
        "Official_Title",
        "Study_Type",
        "Overall_Status",
        "Trial_Completion",
        "Results_Available",
        "Timely_Results_Reporting",
        "Reporting_Category",
        "Days_to_Results_Reporting",
        "Study_Start_Date",
        "Primary_Completion_Date",
        "Study_Completion_Date",
        "Results_First_Posted_Date",
        "Phase",
        "Phase_Category",
        "Allocation",
        "Intervention_Model",
        "Masking",
        "Primary_Purpose",
        "Target_Enrolment",
        "Target_Enrolment_Category",
        "Sponsor_Name",
        "Sponsor_Class",
        "Sponsor_Group",
        "Intervention_Type",
        "Intervention_Types",
        "Countries",
        "Number_of_Countries",
        "Multinational_Study",
        "Number_of_Sites",
        "Number_of_Uganda_Sites",
        "Multisite_Study",
        "Study_Duration_Days",
        "Study_Duration_Months",
        "Study_Duration_Category",
        "Date_Inconsistency",
        "Eligible_Objective_2",
        "Eligible_Results_Analysis",
        "Eligible_Timely_Reporting",
        "Uganda_Involvement",
        "Data_Source"
    ],

    "Label": [

        "ClinicalTrials.gov study identifier",
        "Brief study title",
        "Official study title",
        "Type of clinical study",
        "Overall recruitment/status classification",
        "Clinical trial completion indicator",
        "Results posted availability indicator",
        "Timely results reporting indicator",
        "Results reporting classification",
        "Days between primary completion and results first posted",
        "Study start date",
        "Primary completion date",
        "Overall study completion date",
        "Date results were first posted",
        "Original phase information",
        "Grouped study phase",
        "Study allocation",
        "Intervention model",
        "Masking approach",
        "Primary purpose of study",
        "Planned participant enrolment",
        "Grouped target enrolment",
        "Lead sponsor name",
        "ClinicalTrials.gov sponsor classification",
        "Analytical sponsor grouping",
        "Intervention category",
        "Specific intervention categories",
        "Countries represented in study locations",
        "Number of countries",
        "Whether study is multinational",
        "Number of study facilities/sites",
        "Number of Uganda sites",
        "Whether study is multisite",
        "Study duration in days",
        "Study duration in months",
        "Grouped study duration",
        "Indicator of date inconsistency",
        "Eligibility for Objective Two analysis",
        "Eligibility for results availability analysis",
        "Eligibility for timely reporting analysis",
        "Indicator that Uganda is involved",
        "Original data source"
    ],

    "Type": [

        "String",
        "String",
        "String",
        "Categorical",
        "Categorical",
        "Binary",
        "Binary",
        "Binary",
        "Categorical",
        "Numeric",
        "Date",
        "Date",
        "Date",
        "Date",
        "Categorical",
        "Categorical",
        "Categorical",
        "Categorical",
        "Categorical",
        "Categorical",
        "Numeric",
        "Categorical",
        "String",
        "Categorical",
        "Categorical",
        "Categorical",
        "String",
        "String",
        "Numeric",
        "Binary",
        "Numeric",
        "Numeric",
        "Binary",
        "Numeric",
        "Numeric",
        "Categorical",
        "Binary",
        "Binary",
        "Binary",
        "Binary",
        "Binary",
        "String"
    ],

    "Coding_Notes": [

        "Unique NCT identifier",
        "Text",
        "Text",
        "Interventional or observational",
        "ClinicalTrials.gov overall status",
        "1 = Completed; 0 = other status",
        "1 = results first posted date exists; 0 = unavailable",
        "1 = results posted within 365 days; 0 = more than 365 days; missing if not assessable",
        "Timely, Delayed, Not assessable, or Date inconsistency",
        "Results First Posted Date minus Primary Completion Date",
        "ClinicalTrials.gov study start date",
        "Preferred completion date for reporting analysis",
        "Overall study completion date",
        "ClinicalTrials.gov Results First Posted date",
        "Original ClinicalTrials.gov phase",
        "Grouped phase",
        "Randomized, nonrandomized or other",
        "Parallel, single group, crossover, factorial etc.",
        "None, single, double, triple, quadruple etc.",
        "Treatment, prevention, diagnostic etc.",
        "Number of planned participants",
        "Less than 100, 100 to 499, 500 to 999, 1000 or more",
        "Lead sponsor",
        "ClinicalTrials.gov sponsor class",
        "Industry, Government, Academic/Health Institution, Academic/Other etc.",
        "Drug, biological, device, behavioral, diagnostic etc.",
        "Original intervention type values",
        "Study location countries",
        "Count of unique countries",
        "1 = more than one country; 0 = one country",
        "Count of unique study facilities",
        "Count of Uganda facilities",
        "1 = more than one site; 0 = one site",
        "Completion date minus start date",
        "Duration in days divided by 30.4375",
        "Less than 12, 12 to less than 24, 24 to less than 36, 36 or more months",
        "1 = negative reporting interval; 0 = otherwise",
        "1 = eligible; 0 = otherwise",
        "1 = completed trial; 0 = otherwise",
        "1 = completed trial with valid reporting dates; 0 = otherwise",
        "1 = Uganda involved",
        "ClinicalTrials.gov"
    ]
})


# ================================================================
# 39. CREATE STUDY SUMMARY TABLE
# ================================================================

study_summary = pd.DataFrame({

    "Indicator": [

        "Total registered studies involving Uganda",
        "Completed studies",
        "Non-completed studies",
        "Studies with results available",
        "Studies without results available",
        "Completed studies eligible for timely reporting analysis",
        "Timely results",
        "Delayed results",
        "Studies with negative reporting intervals",
        "Multinational studies",
        "Multisite studies"
    ],

    "Frequency": [

        len(df),

        int(
            (df["Trial_Completion"] == 1).sum()
        ),

        int(
            (df["Trial_Completion"] == 0).sum()
        ),

        int(
            (df["Results_Available"] == 1).sum()
        ),

        int(
            (df["Results_Available"] == 0).sum()
        ),

        int(
            (df["Eligible_Timely_Reporting"] == 1).sum()
        ),

        int(
            (
                df["Timely_Results_Reporting"] == 1
            ).sum()
        ),

        int(
            (
                df["Timely_Results_Reporting"] == 0
            ).sum()
        ),

        int(
            (
                df["Days_to_Results_Reporting"] < 0
            ).sum()
        ),

        int(
            (
                df["Multinational_Study"] == 1
            ).sum()
        ),

        int(
            (
                df["Multisite_Study"] == 1
            ).sum()
        )
    ]
})

study_summary["Percentage"] = (
    study_summary["Frequency"]
    / len(df)
    * 100
).round(2)


# ================================================================
# 40. SAVE OUTPUT DIRECTORY
# ================================================================

output_dir = "/content/Brian_Kabuubi_ClinicalTrials_Uganda"

os.makedirs(
    output_dir,
    exist_ok=True
)


# ================================================================
# 41. SAVE PRINCIPAL ANALYSIS DATASET
# ================================================================

main_file = os.path.join(
    output_dir,
    "Brian_Kabuubi_Uganda_ClinicalTrials_Analysis_Dataset.csv"
)

df.to_csv(
    main_file,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 42. SAVE COMPLETED TRIAL DATASET
# ================================================================

completed_df = df[
    df["Trial_Completion"] == 1
].copy()

completed_file = os.path.join(
    output_dir,
    "Brian_Kabuubi_Completed_Trials_Uganda.csv"
)

completed_df.to_csv(
    completed_file,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 43. SAVE RESULTS ANALYSIS DATASET
# ================================================================

results_analysis_df = df[
    df["Trial_Completion"] == 1
].copy()

results_file = os.path.join(
    output_dir,
    "Brian_Kabuubi_Completed_Trials_Results_Analysis.csv"
)

results_analysis_df.to_csv(
    results_file,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 44. SAVE TIMELY REPORTING DATASET
# ================================================================

timely_reporting_df = df[
    df["Eligible_Timely_Reporting"] == 1
].copy()

timely_file = os.path.join(
    output_dir,
    "Brian_Kabuubi_Timely_Results_Reporting_Analysis.csv"
)

timely_reporting_df.to_csv(
    timely_file,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 45. SAVE DATA DICTIONARY
# ================================================================

dictionary_file = os.path.join(
    output_dir,
    "Brian_Kabuubi_Data_Dictionary.csv"
)

data_dictionary.to_csv(
    dictionary_file,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 46. SAVE MISSING DATA REPORT
# ================================================================

missing_file = os.path.join(
    output_dir,
    "Brian_Kabuubi_Missing_Data_Report.csv"
)

missing_summary.to_csv(
    missing_file,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 47. SAVE DESCRIPTIVE STATISTICS
# ================================================================

descriptive_file = os.path.join(
    output_dir,
    "Brian_Kabuubi_Descriptive_Statistics.csv"
)

descriptive_statistics.to_csv(
    descriptive_file,
    encoding="utf-8-sig"
)


# ================================================================
# 48. SAVE STUDY SUMMARY
# ================================================================

summary_file = os.path.join(
    output_dir,
    "Brian_Kabuubi_Study_Summary.csv"
)

study_summary.to_csv(
    summary_file,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 49. SAVE CROSS TABULATIONS
# ================================================================

study_type_completion.to_csv(
    os.path.join(
        output_dir,
        "Study_Type_by_Completion.csv"
    ),
    encoding="utf-8-sig"
)

sponsor_completion.to_csv(
    os.path.join(
        output_dir,
        "Sponsor_Group_by_Completion.csv"
    ),
    encoding="utf-8-sig"
)

phase_completion.to_csv(
    os.path.join(
        output_dir,
        "Study_Phase_by_Completion.csv"
    ),
    encoding="utf-8-sig"
)

sponsor_results.to_csv(
    os.path.join(
        output_dir,
        "Sponsor_Group_by_Results_Availability.csv"
    ),
    encoding="utf-8-sig"
)

phase_results.to_csv(
    os.path.join(
        output_dir,
        "Study_Phase_by_Results_Availability.csv"
    ),
    encoding="utf-8-sig"
)

if len(sponsor_reporting) > 0:

    sponsor_reporting.to_csv(
        os.path.join(
            output_dir,
            "Sponsor_Group_by_Timely_Reporting.csv"
        ),
        encoding="utf-8-sig"
    )


# ================================================================
# 50. SAVE RAW JSON
#
# This is important for research reproducibility.
# ================================================================

raw_json_file = os.path.join(
    output_dir,
    "ClinicalTrials_Gov_Uganda_Raw_JSON.json"
)

with open(
    raw_json_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        raw_studies,
        f,
        ensure_ascii=False,
        indent=2
    )


# ================================================================
# 51. CREATE README FOR THE RESEARCH PROJECT
# ================================================================

readme_text = f"""
BRIAN KABUUBI
Registration Number: {REG_NO}
College: {COLLEGE}

THESIS TITLE
{THESIS_TITLE}

DATA SOURCE
ClinicalTrials.gov

SEARCH LOCATION
Uganda

DATA EXTRACTION DATE
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

NUMBER OF RECORDS
{len(df)}

PRIMARY OUTCOMES

1. Trial_Completion
   1 = Completed
   0 = Other status

2. Results_Available
   1 = Results First Posted date available
   0 = Results First Posted date unavailable

3. Timely_Results_Reporting
   1 = Results posted within 365 days after Primary Completion Date
   0 = Results posted after 365 days
   Missing = reporting cannot be assessed

IMPORTANT METHODOLOGICAL NOTE

Missing Results First Posted dates are NOT automatically classified
as delayed reporting.

Negative reporting intervals are flagged as Date_Inconsistency and
are excluded from the timely reporting analysis.

PRIMARY COMPLETION DATE is used as the reference date for reporting
timeliness.

OBJECTIVE TWO ANALYSIS

Trial completion can be analysed using:
Chi square test
Binary logistic regression

OBJECTIVE THREE ANALYSIS

Results availability can be analysed using:
Chi square test
Binary logistic regression

Timely results reporting can be analysed using:
Chi square test
Binary logistic regression

Survival analysis may be considered if the number and completeness
of valid reporting dates are sufficient.

RESEARCH REPRODUCIBILITY

The raw JSON file is retained together with the cleaned analytical
CSV files and data dictionary.
"""

readme_file = os.path.join(
    output_dir,
    "README_Brian_Kabuubi_Research_Dataset.txt"
)

with open(
    readme_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        readme_text
    )


# ================================================================
# 52. FINAL OUTPUT REPORT
# ================================================================

print("\n")
print("=" * 80)
print("DATA EXTRACTION AND CLEANING COMPLETED")
print("=" * 80)

print(
    f"Total records: {len(df):,}"
)

print(
    f"Completed trials: "
    f"{int((df['Trial_Completion'] == 1).sum()):,}"
)

print(
    f"Results available: "
    f"{int((df['Results_Available'] == 1).sum()):,}"
)

print(
    f"Eligible for timely reporting analysis: "
    f"{int((df['Eligible_Timely_Reporting'] == 1).sum()):,}"
)

print(
    f"Timely reporting: "
    f"{int((df['Timely_Results_Reporting'] == 1).sum()):,}"
)

print(
    f"Delayed reporting: "
    f"{int((df['Timely_Results_Reporting'] == 0).sum()):,}"
)

print(
    f"Negative reporting intervals flagged: "
    f"{int((df['Days_to_Results_Reporting'] < 0).sum()):,}"
)

print("\nOutput files")
print("-" * 80)

for filename in sorted(
    os.listdir(output_dir)
):

    filepath = os.path.join(
        output_dir,
        filename
    )

    size_mb = (
        os.path.getsize(filepath)
        / (1024 * 1024)
    )

    print(
        f"{filename:<65} "
        f"{size_mb:.2f} MB"
    )


# ================================================================
# 53. DOWNLOAD THE MAIN ANALYSIS DATASET
# ================================================================

print("\nDownloading the principal CSV dataset...")

files.download(
    main_file
)

print(
    "\nThe principal CSV dataset has been sent to your computer."
)


# ================================================================
# 54. OPTIONAL ZIP FILE CONTAINING EVERYTHING
# ================================================================

import shutil

zip_file = shutil.make_archive(
    "/content/Brian_Kabuubi_ClinicalTrials_Uganda",
    "zip",
    output_dir
)

print(
    "\nComplete research data package created:"
)

print(zip_file)

# Download complete research package
files.download(
    zip_file
)

print("\n" + "=" * 80)
print("END OF PIPELINE")
print("=" * 80)

CLINICALTRIALS.GOV DATA EXTRACTION PIPELINE
Student: Brian Kabuubi
Registration Number: 2026 04 47856
College: CEM, Kampala International University
Location: Uganda

ClinicalTrials.gov API information
--------------------------------------------------------------------------------
{
  "apiVersion": "2.0.5",
  "dataTimestamp": "2026-08-28T09:00:06"
}

Records retrieved on this page: 100

Records retrieved on this page: 100

Records retrieved on this page: 100

Records retrieved on this page: 100

Records retrieved on this page: 100

Records retrieved on this page: 100

Records retrieved on this page: 100

Records retrieved on this page: 100

Records retrieved on this page: 100

Records retrieved on this page: 72

TOTAL RAW RECORDS RETRIEVED: 972

Number of NCT IDs retrieved: 972
Number of unique NCT IDs: 972
Duplicate records: 0

Extracting study variables...

Successfully extracted 972 records.

Duplicate check
--------------------------------------------------------------------------

,Variable,Missing_Frequency,Total_Observations,Missing_Percentage
0,Intervention_Model_Description,972,972,100.00
1,Masking_Description,972,972,100.00
2,Results_First_Posted_Date_Type,797,972,82.00
3,Results_First_Posted_Date,797,972,82.00
4,Days_to_Results_Reporting,797,972,82.00
5,Timely_Results_Reporting,797,972,82.00
6,Study_Completion_Date,369,972,37.96
7,Primary_Completion_Date,368,972,37.86
8,Study_Start_Date,318,972,32.72
9,Study_Start_Date_Type,278,972,28.60



Missingness among principal analytical variables


,Variable,Missing_Frequency,Total_Observations,Missing_Percentage
3,Results_First_Posted_Date,797,972,82.00
5,Timely_Results_Reporting,797,972,82.00
7,Primary_Completion_Date,368,972,37.86
13,Study_Duration_Months,11,972,1.13
16,Target_Enrolment,1,972,0.10
20,Trial_Completion,0,972,0.00
22,Study_Type,0,972,0.00
23,Results_Available,0,972,0.00
25,Intervention_Model,0,972,0.00
26,Allocation,0,972,0.00



DATA QUALITY CHECKS


...

<IPython.core.display.Javascript object>


The principal CSV dataset has been sent to your computer.

Complete research data package created:
/content/Brian_Kabuubi_ClinicalTrials_Uganda.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


END OF PIPELINE
